# WaterWise-Style Optimization Case Study

This notebook is a simple, energy-relevant linear optimization demo for workshop use.

## Objectives
- Load a small water-allocation dataset
- Formulate the business problem as an optimization model
- Solve it with `PuLP`
- Interpret the resulting allocation plan
- Connect the toy model to larger oil and gas optimization workflows

## Case Summary
A completions team must allocate water from produced-water, brackish, and freshwater sources to frac pads while minimizing cost and respecting operational constraints.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import pulp
import seaborn as sns

sns.set_theme(style="whitegrid")

candidate_dirs = [
    Path("../sample-data/optimization_waterwise"),
    Path("training/sample-data/optimization_waterwise"),
    Path("sample-data/optimization_waterwise"),
]

data_dir = next((path for path in candidate_dirs if path.exists()), None)
if data_dir is None:
    raise FileNotFoundError("Could not locate optimization_waterwise sample data.")

sources = pd.read_csv(data_dir / "water_sources.csv")
demands = pd.read_csv(data_dir / "demand_pads.csv")
links = pd.read_csv(data_dir / "transport_links.csv")
params = pd.read_csv(data_dir / "scenario_parameters.csv")
param_map = dict(zip(params["parameter"], params["value"]))

sources, demands, links, params

## Step 1: Translate The Problem

We want to decide how much water to send from each source to each pad.

### Decision variables
- `flow[source, pad]`: barrels allocated from a source to a pad
- `shortage[pad]`: unmet demand, heavily penalized

### Objective
Minimize total sourcing cost, transport cost, and any shortage penalty.

### Constraints
- each source has limited supply
- each route has a capacity limit
- each pad must be supplied, unless shortage is used
- freshwater usage must stay below a cap
- produced-water reuse must meet a minimum target

In [ ]:
source_map = sources.set_index("source").to_dict("index")
demand_map = demands.set_index("pad").to_dict("index")
link_map = {(row.source, row.pad): row for row in links.itertuples(index=False)}

model = pulp.LpProblem("WaterWise_Training_Case", pulp.LpMinimize)

flow = {
    (source, pad): pulp.LpVariable(f"flow__{source}__{pad}", lowBound=0)
    for source, pad in link_map
}
shortage = {
    pad: pulp.LpVariable(f"shortage__{pad}", lowBound=0)
    for pad in demand_map
}

model += (
    pulp.lpSum(
        flow[source, pad]
        * (source_map[source]["source_cost_per_bbl"] + link_map[source, pad].transport_cost_per_bbl)
        for source, pad in flow
    )
    + pulp.lpSum(shortage[pad] * demand_map[pad]["shortage_penalty_per_bbl"] for pad in shortage),
    "total_cost",
)

for source in source_map:
    model += (
        pulp.lpSum(flow[s, p] for s, p in flow if s == source) <= source_map[source]["supply_bbl"],
        f"supply_limit__{source}",
    )

for pad in demand_map:
    model += (
        pulp.lpSum(flow[s, p] for s, p in flow if p == pad) + shortage[pad] == demand_map[pad]["demand_bbl"],
        f"demand_balance__{pad}",
    )

for (source, pad), link in link_map.items():
    model += (flow[source, pad] <= link.max_capacity_bbl, f"route_capacity__{source}__{pad}")

freshwater_sources = [source for source, row in source_map.items() if row["source_type"] == "freshwater"]
model += (
    pulp.lpSum(flow[s, p] for s, p in flow if s in freshwater_sources) <= float(param_map["freshwater_cap_bbl"]),
    "freshwater_cap",
)

produced_sources = [source for source, row in source_map.items() if row["source_type"] == "produced_water"]
model += (
    pulp.lpSum(flow[s, p] for s, p in flow if s in produced_sources) >= float(param_map["min_produced_water_reuse_bbl"]),
    "produced_water_reuse_target",
)

solver = pulp.PULP_CBC_CMD(msg=False)
model.solve(solver)
print("Solver status:", pulp.LpStatus[model.status])
print("Objective value: ${:,.2f}".format(pulp.value(model.objective)))

In [ ]:
allocation_rows = []
for (source, pad), var in flow.items():
    value = float(var.value() or 0.0)
    if value > 1e-6:
        allocation_rows.append(
            {
                "source": source,
                "pad": pad,
                "source_type": source_map[source]["source_type"],
                "allocated_bbl": round(value, 2),
            }
        )

allocation_df = pd.DataFrame(allocation_rows).sort_values(["pad", "source"])
pad_summary_df = (
    allocation_df.groupby("pad", as_index=False)["allocated_bbl"].sum()
    .merge(demands, on="pad", how="right")
    .fillna({"allocated_bbl": 0.0})
)
pad_summary_df["shortage_bbl"] = pad_summary_df["pad"].map(lambda pad: float(shortage[pad].value() or 0.0))
pad_summary_df["fulfillment_pct"] = 100 * pad_summary_df["allocated_bbl"] / pad_summary_df["demand_bbl"]

source_summary_df = allocation_df.groupby(["source", "source_type"], as_index=False)["allocated_bbl"].sum()

allocation_df
pad_summary_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=source_summary_df, x="source", y="allocated_bbl", hue="source_type", ax=axes[0])
axes[0].set_title("Allocated Water By Source")
axes[0].set_xlabel("Source")
axes[0].set_ylabel("Allocated bbl")
axes[0].tick_params(axis="x", rotation=25)

plot_df = allocation_df.copy()
sns.barplot(data=plot_df, x="pad", y="allocated_bbl", hue="source_type", ax=axes[1])
axes[1].set_title("Pad Supply Mix")
axes[1].set_xlabel("Pad")
axes[1].set_ylabel("Allocated bbl")

plt.tight_layout()
plt.show()

produced_used = source_summary_df.loc[source_summary_df["source_type"] == "produced_water", "allocated_bbl"].sum()
freshwater_used = source_summary_df.loc[source_summary_df["source_type"] == "freshwater", "allocated_bbl"].sum()
total_used = source_summary_df["allocated_bbl"].sum()
print(f"Produced water reused: {produced_used:,.0f} bbl")
print(f"Freshwater used: {freshwater_used:,.0f} bbl")
print(f"Produced-water share: {100 * produced_used / total_used:0.1f}%")

## Reflection Questions

1. Why is this an optimization problem instead of a machine-learning problem?
2. What are the decision variables in plain engineering language?
3. What happens if the freshwater cap becomes tighter?
4. What happens if one produced-water route loses capacity?
5. Which real field constraints would you add next?

## Practical Extensions
- Add trucking and pipeline choices separately
- Add water quality compatibility constraints
- Add intermediate storage tanks
- Add multiple time periods
- Add integer decisions such as whether a route is used at all

## Bigger Picture
This same structure appears in:
- water management
- sand logistics
- drilling and completions scheduling
- production routing and allocation
- CO2 transport-network design